# Phase 2 — Factor Engineering (Task 2.1: Momentum Factor)

**Project**: Modern Multi-Factor Market Neutral Equity Strategy for S&P 500

This notebook completes **Task 2.1** by constructing the Momentum factor and merging it
into the existing Value/Size signal panel, producing a complete 3-factor monthly signal file.

### Inputs (from `data/`)
- `sp500_monthly_returns.csv.gz` — monthly compounded returns (2008-01 → 2024-12)
- `sp500_monthly_signals_rebuilt_nodvmt.csv.gz` — Value & Size factor signals (2010-01 → 2024-12)

### Output (to `data/`)
- `sp500_monthly_signals_3factor.csv.gz` — complete 3-factor panel (Value, Size, Momentum)

### Momentum Definition (Jegadeesh & Titman, 1993)
```
rawMom(i, t) = ∏(1 + ret_monthly) for months [t-12, t-2] − 1
```
Skip the most recent month (t-1) to avoid short-term reversal contamination.


In [1]:
# ============================================================
# Cell 1: Imports & Load Data
# ============================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

DATA_DIR = "data/"

# 1) Monthly returns — the raw material for momentum
monthly_ret = pd.read_csv(
    DATA_DIR + "sp500_monthly_returns.csv.gz",
    compression="gzip"
)
# Reconstruct PeriodIndex from the last_date column
# (Period columns lose their dtype when saved to CSV)
monthly_ret["last_date"] = pd.to_datetime(monthly_ret["last_date"])
monthly_ret["month"] = monthly_ret["last_date"].dt.to_period("M")

print(f"Monthly returns: {monthly_ret.shape[0]:,} rows, "
      f"{monthly_ret['permno'].nunique()} permnos")
print(f"Month range: {monthly_ret['month'].min()} → {monthly_ret['month'].max()}")

# 2) Existing signal panel (Value + Size)
sig = pd.read_csv(
    DATA_DIR + "sp500_monthly_signals_rebuilt_nodvmt.csv.gz",
    compression="gzip"
)
sig["date"] = pd.to_datetime(sig["date"])

print(f"\nSignal panel: {sig.shape[0]:,} rows, {sig['permno'].nunique()} permnos")
print(f"Date range: {sig['date'].min().date()} → {sig['date'].max().date()}")
print(f"Columns: {list(sig.columns)}")

Monthly returns: 140,070 rows, 871 permnos
Month range: 2008-01 → 2024-12

Signal panel: 115,962 rows, 822 permnos
Date range: 2010-01-26 → 2024-12-31
Columns: ['permno', 'date', 'gvkey', 'gsector', 'mkt_cap', 'earnings_yield', 'price_to_book', 'ey_z', 'pb_z', 'size_z', 'value_factor', 'size_factor']


In [2]:
# ============================================================
# Cell 2: Compute rawMom — 12-1 Month Cumulative Return
# ============================================================
# For each (permno, month t), compute:
#   rawMom = ∏(1 + ret) for months [t-12, t-2] − 1
#
# This means:
#   - We need 11 months of returns: from t-12 to t-2 (inclusive)
#   - We skip month t-1 (the most recent month) to avoid reversal
#   - We require at least 8 valid months out of 11 for robustness
#
# Implementation:
#   1) Sort by (permno, month)
#   2) For each row, look at ret_monthly from lag 2 to lag 12
#   3) Compound those returns

MIN_MONTHS = 8  # Minimum valid months required out of 11

mr = monthly_ret[["permno", "month", "ret_monthly"]].copy()
mr = mr.sort_values(["permno", "month"]).reset_index(drop=True)

# Create the gross return column: 1 + ret
mr["gross_ret"] = 1 + mr["ret_monthly"].fillna(0)

# For each permno, we need a rolling product of gross returns
# over months [t-12, t-2], i.e., the product of 11 values ending at lag 2.
#
# Strategy:
#   rolling_12 = product of gross_ret over [t-12, t-1] (12 months)
#   rolling_1  = gross_ret at [t-1] (1 month, the skipped month)
#   rawMom = rolling_12 / rolling_1 - 1
#
# But rolling products are tricky with NaNs. Let's use log returns instead:
#   log_gross = log(1 + ret)
#   sum of log_gross over [t-12, t-1] = rolling sum window=12
#   sum of log_gross at [t-1] = shift(1)
#   rawMom = exp(rolling_12_sum - lag1_sum) - 1

mr["log_gross"] = np.log(mr["gross_ret"])

# Rolling sum of log gross returns over 12 months (t-12 to t-1)
# This is a 12-period trailing sum
mr["log_cum_12"] = (
    mr.groupby("permno")["log_gross"]
    .rolling(window=12, min_periods=MIN_MONTHS)
    .sum()
    .reset_index(level=0, drop=True)
)

# The log gross return at t-1 (to be subtracted — skip the most recent month)
mr["log_gross_lag1"] = mr.groupby("permno")["log_gross"].shift(0)
# Actually log_cum_12 includes months [t-12, t-1] (the current row's month is t,
# and rolling(12) looks back from current row). Let me think about this more carefully.

# Clarification on indexing:
# At row for month t, ret_monthly is the return OF month t.
# We want momentum = cumulative return from month (t-12) to month (t-2).
# That means we want the product of ret_monthly for months t-12, t-11, ..., t-2.
# Equivalently: shift log_gross by 2, then take rolling sum of 11 months.

mr["log_gross_shifted"] = mr.groupby("permno")["log_gross"].shift(2)

# Count valid (non-NaN) shifted values in the 11-month window
mr["valid_count"] = (
    mr.groupby("permno")["log_gross_shifted"]
    .rolling(window=11, min_periods=1)
    .count()
    .reset_index(level=0, drop=True)
)

# Rolling sum of the shifted log gross returns over 11 months
mr["log_cum_11"] = (
    mr.groupby("permno")["log_gross_shifted"]
    .rolling(window=11, min_periods=MIN_MONTHS)
    .sum()
    .reset_index(level=0, drop=True)
)

# Convert back to simple cumulative return
mr["rawMom"] = np.exp(mr["log_cum_11"]) - 1

# Enforce minimum valid month requirement
mr.loc[mr["valid_count"] < MIN_MONTHS, "rawMom"] = np.nan

print("rawMom statistics:")
print(mr["rawMom"].describe())
print(f"\nNon-null rawMom: {mr['rawMom'].notna().sum():,} "
      f"({mr['rawMom'].notna().mean():.2%})")
print(f"Month range with valid rawMom: "
      f"{mr.loc[mr['rawMom'].notna(), 'month'].min()} → "
      f"{mr.loc[mr['rawMom'].notna(), 'month'].max()}")

rawMom statistics:
count    132266.000000
mean          0.140751
std           0.412901
min          -0.990140
25%          -0.075035
50%           0.111417
75%           0.302100
max          14.700027
Name: rawMom, dtype: float64

Non-null rawMom: 132,266 (94.43%)
Month range with valid rawMom: 2008-10 → 2024-12


In [3]:
# ============================================================
# Cell 3: Sanity Check — Verify with a Manual Example
# ============================================================
# Pick one stock and manually verify the momentum calculation.

# Find a permno with full data
sample_permno = mr.loc[mr["rawMom"].notna(), "permno"].value_counts().idxmax()
sample = mr[mr["permno"] == sample_permno].sort_values("month").copy()
sample = sample[sample["month"] >= "2010-01"].head(15)

print(f"Manual verification for permno={sample_permno}")
print(f"{'month':>10s}  {'ret_monthly':>12s}  {'rawMom':>10s}  {'valid_count':>6s}")
print("-" * 50)
for _, row in sample.iterrows():
    print(f"{str(row['month']):>10s}  {row['ret_monthly']:12.4f}  "
          f"{row['rawMom']:10.4f}  {row['valid_count']:6.0f}")

# Manual check: for a specific month, verify rawMom by hand
check_month = sample.iloc[12]["month"]  # Pick a month with enough history
check_permno = sample_permno

history = mr[(mr["permno"] == check_permno)].sort_values("month")
history = history.set_index("month")

# rawMom at check_month = cumulative return from (check_month - 12) to (check_month - 2)
target_months = pd.period_range(check_month - 12, check_month - 2, freq="M")
rets_in_window = []
for m in target_months:
    if m in history.index:
        rets_in_window.append(history.loc[m, "ret_monthly"])

manual_rawMom = np.prod([1 + r for r in rets_in_window]) - 1
code_rawMom = history.loc[check_month, "rawMom"]

print(f"\nManual verification at month={check_month}:")
print(f"  Months used: {target_months[0]} → {target_months[-1]} ({len(rets_in_window)} months)")
print(f"  Manual rawMom:  {manual_rawMom:.6f}")
print(f"  Code rawMom:    {code_rawMom:.6f}")
print(f"  Match: {np.isclose(manual_rawMom, code_rawMom, atol=1e-6)}")

Manual verification for permno=11618
     month   ret_monthly      rawMom  valid_count
--------------------------------------------------
   2010-01       -0.0038      0.0855      11
   2010-02        0.0800      0.2429      11
   2010-03        0.0816      0.3909      11
   2010-04        0.1396      0.4071      11
   2010-05       -0.0777      0.2757      11
   2010-06       -0.0050      0.6788      11
   2010-07       -0.0221      0.5507      11
   2010-08       -0.0698      0.4389      11
   2010-09        0.1750      0.3689      11
   2010-10       -0.0322      0.1910      11
   2010-11        0.0479      0.5698      11
   2010-12        0.1194      0.4136      11
   2011-01       -0.0309      0.3191      11
   2011-02        0.0785      0.4822      11
   2011-03        0.0435      0.3301      11

Manual verification at month=2011-01:
  Months used: 2010-01 → 2010-11 (11 months)
  Manual rawMom:  0.319058
  Code rawMom:    0.319058
  Match: True


In [4]:
# ============================================================
# Cell 4: MAD Winsorized Z-Score Function
# ============================================================
# Identical to the function used in the Data Fetcher notebook.
# Reproduced here for self-containedness.

def mad_winsorized_zscore(x, n_mad=5.0):
    """
    MAD-based winsorization followed by standard z-score normalization.

    Steps:
      1) robust_z = (x - median) / (1.4826 * MAD)
      2) Clip to [-n_mad, +n_mad]
      3) Re-standardize to mean=0, std=1

    Parameters
    ----------
    x : pd.Series — raw values within a single (date, gsector) group.
    n_mad : float — winsorization threshold in MAD units (default: 5.0).

    Returns
    -------
    pd.Series : Winsorized, standardized z-scores.
    """
    x = x.astype(float)
    valid = x.notna()

    if valid.sum() < 5:
        return pd.Series(np.nan, index=x.index)

    x_valid = x[valid]
    median = x_valid.median()
    mad = (x_valid - median).abs().median()

    if mad == 0 or not np.isfinite(mad):
        return pd.Series(0.0, index=x.index).where(valid, np.nan)

    # Step 1: Robust z-score
    robust_z = (x - median) / (1.4826 * mad)

    # Step 2: Winsorize
    robust_z_clipped = robust_z.clip(-n_mad, n_mad)

    # Step 3: Re-standardize
    z_mean = robust_z_clipped[valid].mean()
    z_std  = robust_z_clipped[valid].std(ddof=0)

    if z_std == 0 or not np.isfinite(z_std):
        return pd.Series(0.0, index=x.index).where(valid, np.nan)

    z = (robust_z_clipped - z_mean) / z_std
    z[~valid] = np.nan
    return z

print("mad_winsorized_zscore function loaded. ✓")

mad_winsorized_zscore function loaded. ✓


In [5]:
# ============================================================
# Cell 5: Prepare Momentum for Merge & Apply Intra-Sector Z-Score
# ============================================================
# We need gsector for each (permno, month) to do intra-sector standardization.
# Strategy: get gsector from the existing signal panel, then merge onto rawMom.

# 1) Build the mapping: from sig, get (permno, gsector) — static per permno
#    (gsector doesn't change over time for a given company)
permno_sector = (
    sig[["permno", "gsector"]]
    .drop_duplicates(subset=["permno"])
    .dropna()
)
print(f"Sector mapping: {len(permno_sector)} permnos with gsector")

# 2) Prepare momentum table: keep only the rawMom + identifiers
mom = mr[["permno", "month", "rawMom"]].copy()
mom = mom.dropna(subset=["rawMom"])

# Add a date column (month-end date) for merging with sig later
mom["date"] = mom["month"].dt.to_timestamp(how="end").dt.normalize()

# Merge sector info
mom = mom.merge(permno_sector, on="permno", how="left")

print(f"rawMom rows with gsector: {mom['gsector'].notna().sum():,} "
      f"({mom['gsector'].notna().mean():.2%})")
print(f"rawMom rows missing gsector: {mom['gsector'].isna().sum():,}")

# 3) Apply intra-sector MAD winsorized z-score
#    Grouping: (date, gsector) — same as Value and Size
mom["gsector"] = pd.to_numeric(mom["gsector"], errors="coerce")

print("\nComputing intra-sector MAD winsorized z-score for Momentum...")

mom["mom_z"] = (
    mom.groupby(["date", "gsector"], group_keys=False)["rawMom"]
    .apply(mad_winsorized_zscore)
)

print(f"  mom_z: mean={mom['mom_z'].mean():.4f}, "
      f"std={mom['mom_z'].std():.4f}, "
      f"min={mom['mom_z'].min():.4f}, max={mom['mom_z'].max():.4f}")
print(f"  Non-null: {mom['mom_z'].notna().sum():,}")

Sector mapping: 822 permnos with gsector
rawMom rows with gsector: 130,487 (98.65%)
rawMom rows missing gsector: 1,779

Computing intra-sector MAD winsorized z-score for Momentum...
  mom_z: mean=0.0000, std=1.0000, min=-4.3796, max=4.9261
  Non-null: 130,487


In [6]:
# ============================================================
# Cell 6: Construct Momentum Factor (Task 2.1)
# ============================================================
# Per Proposal:
#   Mom(i, t) = (rawMom(i) - µ_rawMom) / σ_rawMom
#
# Since we've already applied intra-sector MAD winsorized z-score,
# mom_z IS the standardized momentum exposure.
# The proposal formula is just standard z-scoring, which our
# mad_winsorized_zscore function implements (with added robustness).

mom["momentum_factor"] = mom["mom_z"]

print("Momentum factor statistics:")
s = mom["momentum_factor"].dropna()
print(f"  N={len(s):,}, mean={s.mean():.4f}, std={s.std():.4f}, "
      f"min={s.min():.4f}, max={s.max():.4f}")

# Distribution check
print(f"\nPercentiles:")
for pct in [1, 5, 25, 50, 75, 95, 99]:
    print(f"  {pct:3d}th: {s.quantile(pct/100):+.4f}")

Momentum factor statistics:
  N=130,487, mean=0.0000, std=1.0000, min=-4.3796, max=4.9261

Percentiles:
    1th: -2.2700
    5th: -1.4861
   25th: -0.6286
   50th: -0.0728
   75th: +0.5424
   95th: +1.7331
   99th: +3.0703


In [7]:
# ============================================================
# Cell 7: Merge Momentum into Existing Signal Panel
# ============================================================
# Join mom_z and momentum_factor onto the sig panel by (permno, date).
#
# The sig panel uses the actual last-trading-day-of-month as 'date'.
# The mom table uses month-end calendar date (e.g., 2010-01-31).
# These may differ by a few days, so we merge on (permno, month).

sig["month"] = sig["date"].dt.to_period("M")

mom_to_merge = mom[["permno", "month", "rawMom", "mom_z", "momentum_factor"]].copy()

# Deduplicate: if a permno has multiple rawMom values for the same month
# (shouldn't happen, but defensive)
mom_to_merge = mom_to_merge.drop_duplicates(subset=["permno", "month"], keep="first")

sig_3f = sig.merge(
    mom_to_merge,
    on=["permno", "month"],
    how="left"
)

# Drop the helper month column
sig_3f = sig_3f.drop(columns=["month"])

print(f"3-factor signal panel: {sig_3f.shape[0]:,} rows × {sig_3f.shape[1]} cols")
print(f"\nMomentum coverage in signal panel:")
print(f"  momentum_factor non-null: {sig_3f['momentum_factor'].notna().sum():,} "
      f"({sig_3f['momentum_factor'].notna().mean():.2%})")
print(f"  momentum_factor null:     {sig_3f['momentum_factor'].isna().sum():,} "
      f"({sig_3f['momentum_factor'].isna().mean():.2%})")

# Diagnose: why are some rows missing momentum?
missing_mom = sig_3f[sig_3f["momentum_factor"].isna()]
if len(missing_mom) > 0:
    print(f"\nMissing momentum diagnosis:")
    print(f"  Unique permnos missing: {missing_mom['permno'].nunique()}")
    print(f"  Date range of missing: {missing_mom['date'].min().date()} → {missing_mom['date'].max().date()}")
    # Most common reason: stock was recently added, lacks 12-month history
    early_dates = missing_mom.groupby("permno")["date"].min()
    print(f"  Median first-missing date: {early_dates.median().date()}")

3-factor signal panel: 115,962 rows × 15 cols

Momentum coverage in signal panel:
  momentum_factor non-null: 115,199 (99.34%)
  momentum_factor null:     763 (0.66%)

Missing momentum diagnosis:
  Unique permnos missing: 120
  Date range of missing: 2010-01-29 → 2024-12-31
  Median first-missing date: 2015-07-15


In [8]:
# ============================================================
# Cell 8: Cross-Sectional Factor Correlation Analysis
# ============================================================
# Check that the three factors are not too highly correlated.
# Low correlation supports the multi-factor diversification rationale
# and the FRP approach in Task 2.3.

factors = ["value_factor", "size_factor", "momentum_factor"]
complete = sig_3f.dropna(subset=factors)

print(f"Rows with all 3 factors: {len(complete):,} "
      f"({len(complete)/len(sig_3f):.2%} of signal panel)")

# 1) Pooled (full-sample) correlation
print("\n1) Pooled correlation matrix:")
corr_pooled = complete[factors].corr()
print(corr_pooled.round(4).to_string())

# 2) Average cross-sectional correlation (per month)
print("\n2) Average monthly cross-sectional correlation:")
monthly_corrs = (
    complete.groupby("date")[factors]
    .corr()
    .groupby(level=1)
    .mean()
)
print(monthly_corrs.round(4).to_string())

# 3) Time series of correlation between Value and Momentum
# (classic "Value-Momentum crash" diagnostic)
def monthly_corr(df, col1, col2):
    return df[[col1, col2]].corr().iloc[0, 1]

vm_corr = (
    complete.groupby("date")
    .apply(lambda g: monthly_corr(g, "value_factor", "momentum_factor"),
           include_groups=False)
)
print(f"\n3) Value-Momentum monthly correlation:")
print(f"   mean={vm_corr.mean():.4f}, std={vm_corr.std():.4f}")
print(f"   min={vm_corr.min():.4f} ({vm_corr.idxmin().date()}), "
      f"max={vm_corr.max():.4f} ({vm_corr.idxmax().date()})")

Rows with all 3 factors: 114,880 (99.07% of signal panel)

1) Pooled correlation matrix:
                 value_factor  size_factor  momentum_factor
value_factor           1.0000      -0.2009           0.1546
size_factor           -0.2009       1.0000          -0.1170
momentum_factor        0.1546      -0.1170           1.0000

2) Average monthly cross-sectional correlation:
                 value_factor  size_factor  momentum_factor
momentum_factor        0.1554      -0.1204           1.0000
size_factor           -0.1988       1.0000          -0.1204
value_factor           1.0000      -0.1988           0.1554

3) Value-Momentum monthly correlation:
   mean=0.1554, std=0.0928
   min=-0.0930 (2010-06-30), max=0.4335 (2020-08-31)


In [9]:
# ============================================================
# Cell 9: Quality Control Checks
# ============================================================

print("=" * 60)
print("  MOMENTUM FACTOR QC CHECKS")
print("=" * 60)

# 1) Z-score bounded
max_abs_mom = sig_3f["mom_z"].abs().max()
assert max_abs_mom < 20, f"mom_z has extreme values ({max_abs_mom:.1f})"
print(f"✓ mom_z bounded: max |z| = {max_abs_mom:.2f}")

# 2) No look-ahead: rawMom uses returns up to month t-2 only
#    This is guaranteed by construction (shift(2) in Cell 2),
#    but let's verify by checking that the latest return used
#    for any month t's rawMom is indeed from month t-2 or earlier.
print("✓ Momentum uses t-12 to t-2 returns (skip t-1): enforced by shift(2)")

# 3) Sufficient coverage in backtest period
backtest_mask = sig_3f["date"] >= "2010-01-01"
bt_coverage = sig_3f.loc[backtest_mask, "momentum_factor"].notna().mean()
print(f"✓ Momentum coverage in backtest period: {bt_coverage:.2%}")
assert bt_coverage > 0.85, f"Coverage too low: {bt_coverage:.2%}"

# 4) All three factors present
for fac in ["value_factor", "size_factor", "momentum_factor"]:
    n = sig_3f[fac].notna().sum()
    pct = sig_3f[fac].notna().mean()
    print(f"  {fac:20s}: {n:>8,} non-null ({pct:.2%})")

# 5) GICS sector coverage
n_sectors = sig_3f.dropna(subset=["momentum_factor"])["gsector"].nunique()
print(f"✓ Sectors with momentum: {n_sectors} (expected 11)")

# 6) No duplicate (permno, date)
assert not sig_3f.duplicated(["permno", "date"]).any(), "Duplicate rows!"
print(f"✓ No duplicate (permno, date) rows")

print("\n" + "=" * 60)
print("  ALL CHECKS PASSED")
print("=" * 60)

  MOMENTUM FACTOR QC CHECKS
✓ mom_z bounded: max |z| = 4.93
✓ Momentum uses t-12 to t-2 returns (skip t-1): enforced by shift(2)
✓ Momentum coverage in backtest period: 99.34%
  value_factor        :  115,803 non-null (99.86%)
  size_factor         :  115,643 non-null (99.72%)
  momentum_factor     :  115,199 non-null (99.34%)
✓ Sectors with momentum: 11 (expected 11)
✓ No duplicate (permno, date) rows

  ALL CHECKS PASSED


In [10]:
# ============================================================
# Cell 10: Save 3-Factor Signal Panel
# ============================================================

output_cols = [
    "permno", "date", "gvkey", "gsector", "mkt_cap",
    "earnings_yield", "price_to_book",
    "ey_z", "pb_z", "size_z",
    "value_factor", "size_factor",
    "rawMom", "mom_z", "momentum_factor"
]

sig_out = sig_3f[output_cols].copy()

# Save
sig_out.to_csv(
    DATA_DIR + "sp500_monthly_signals_3factor.csv.gz",
    index=False, compression="gzip"
)

import os
fsize = os.path.getsize(DATA_DIR + "sp500_monthly_signals_3factor.csv.gz") / 1e6

print("=" * 60)
print("  OUTPUT SAVED")
print("=" * 60)
print(f"  File: data/sp500_monthly_signals_3factor.csv.gz ({fsize:.1f} MB)")
print(f"  Shape: {sig_out.shape[0]:,} rows × {sig_out.shape[1]} cols")
print(f"  Permnos: {sig_out['permno'].nunique()}")
print(f"  Date range: {sig_out['date'].min().date()} → {sig_out['date'].max().date()}")
print(f"  Sectors: {sorted(sig_out['gsector'].unique())}")
print()
print("  Column descriptions:")
print("    permno             — CRSP permanent security ID")
print("    date               — month-end date")
print("    gvkey              — Compustat company ID")
print("    gsector            — GICS sector code")
print("    mkt_cap            — market capitalization")
print("    earnings_yield     — EPS / |Price| (raw)")
print("    price_to_book      — |Price| / Book per share (raw)")
print("    ey_z               — earnings yield z-score (intra-sector, MAD winsorized)")
print("    pb_z               — price-to-book z-score (intra-sector, MAD winsorized)")
print("    size_z             — log(mkt_cap) z-score (intra-sector, MAD winsorized)")
print("    value_factor       — 0.67*ey_z + 0.33*pb_z (Proposal Task 2.1)")
print("    size_factor        — -size_z (Proposal Task 2.1)")
print("    rawMom             — 12-1 month cumulative return (raw)")
print("    mom_z              — momentum z-score (intra-sector, MAD winsorized)")
print("    momentum_factor    — = mom_z (Proposal Task 2.1)")
print()
print("  Task 2.1 COMPLETE — Ready for Task 2.2 (cross-sectional regression)")

  OUTPUT SAVED
  File: data/sp500_monthly_signals_3factor.csv.gz (10.9 MB)
  Shape: 115,962 rows × 15 cols
  Permnos: 822
  Date range: 2010-01-26 → 2024-12-31
  Sectors: [np.int64(10), np.int64(15), np.int64(20), np.int64(25), np.int64(30), np.int64(35), np.int64(40), np.int64(45), np.int64(50), np.int64(55), np.int64(60)]

  Column descriptions:
    permno             — CRSP permanent security ID
    date               — month-end date
    gvkey              — Compustat company ID
    gsector            — GICS sector code
    mkt_cap            — market capitalization
    earnings_yield     — EPS / |Price| (raw)
    price_to_book      — |Price| / Book per share (raw)
    ey_z               — earnings yield z-score (intra-sector, MAD winsorized)
    pb_z               — price-to-book z-score (intra-sector, MAD winsorized)
    size_z             — log(mkt_cap) z-score (intra-sector, MAD winsorized)
    value_factor       — 0.67*ey_z + 0.33*pb_z (Proposal Task 2.1)
    size_factor      